# Kestrel Milestone 1: Raw Data Profiling

This notebook profiles the original nine M1 inputs before cleaning. It deliberately excludes `activity_part1.parquet` through `activity_part6.parquet` and does not use the M2-only stream or incident-label files. Run from the repository root or from `M1/`.

In [11]:
REPO = Path.cwd()
if REPO.name == 'M1':
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
M1 = REPO / 'M1'

# EDIT THIS LINE: folder that contains the nine raw files
DATA_ROOT = Path(r'C:\Users\LENOVO\Downloads\kestrel-telemetry-pack-v1')
ROOT = DATA_ROOT

## Activity Parquet: Before Cleaning

In [12]:
parquet_path = ROOT / 'agent_activity.parquet'
parquet_file = pq.ParquetFile(parquet_path)
activity = parquet_file.read().to_pandas()
print('Parquet schema:\n', parquet_file.schema_arrow)
print('Rows:', parquet_file.metadata.num_rows, 'row groups:', parquet_file.num_row_groups)
print('Full-row duplicates:', activity.duplicated().sum())
print('Duplicate event_id rows:', activity['event_id'].duplicated().sum())
print('Null rates:\n', activity.isna().mean().sort_values(ascending=False).to_string())
print('Cardinalities:\n', activity.nunique(dropna=False).to_string())
print('Event types:\n', activity['event_type'].value_counts(dropna=False).to_string())
print('Statuses:\n', activity['status'].value_counts(dropna=False).to_string())
for column in ['event_time', 'ingest_time', 'finished_at']:
    raw = activity[column].dropna().astype(str)
    epoch_ms = raw.str.fullmatch(r'[0-9]{13}').sum()
    explicit_zone = raw.str.contains(r'(?:Z|[+-][0-9]{2}:?[0-9]{2})$', regex=True).sum()
    parsed = pd.to_datetime(raw, format='mixed', utc=True, errors='coerce', dayfirst=True)
    print(f'{column}: present={len(raw):,}; epoch_ms={epoch_ms:,}; explicit_zone={explicit_zone:,}; parse_fail={parsed.isna().sum():,}; min={parsed.min()}; max={parsed.max()}')

Parquet schema:
 event_id: string
event_time: string
ingest_time: string
sequence_number: int64
event_type: string
session_id: string
tenant_id: string
agent_id: string
release_version: string
sandbox_instance_id: string
status: string
latency_ms: int64
turn_index: int64
model_id: string
content_source: string
stated_goal: string
injection_indicators: string
input_tokens: int64
output_tokens: int64
finish_reason: string
tool_id: string
tool_version: string
finished_at: string
retry_number: int64
error_class: string
target_resource: string
bytes_out: int64
task_id: string
task_category: string
expected_tool_categories: string
Rows: 937752 row groups: 1
Full-row duplicates: 0
Duplicate event_id rows: 7427
Null rates:
 error_class                 0.967920
bytes_out                   0.953826
task_id                     0.892500
task_category               0.892500
expected_tool_categories    0.892500
tool_id                     0.655115
tool_version                0.655115
target_resource

## Reference CSVs and Model Pricing: Before Cleaning

In [13]:
def profile_csv(filename, natural_key=None):
    frame = pd.read_csv(ROOT / filename)
    print(f'\n{filename}: rows={len(frame):,}, columns={len(frame.columns)}')
    print('null rates:\n', frame.isna().mean().to_string())
    print('cardinalities:\n', frame.nunique(dropna=False).to_string())
    print('exact duplicate rows:', frame.duplicated().sum())
    if natural_key:
        print('natural-key duplicate rows:', frame.duplicated(natural_key).sum(), 'key:', natural_key)
    return frame

tenants = profile_csv('tenants.csv', ['tenant_id'])
agents = profile_csv('agents.csv', ['agent_id', 'release_version'])
tools = profile_csv('tools.csv', ['tool_id', 'tool_version'])
pricing = profile_csv('model_pricing.csv', ['model_id', 'effective_from'])
print('Agent-version frequency:\n', agents.groupby('agent_id')['release_version'].agg(list).to_string())
print('Tool identifiers and versions:\n', tools[['tool_id', 'tool_version', 'tool_name']].to_string(index=False))
print('Model price rows:\n', pricing.to_string(index=False))


tenants.csv: rows=6, columns=8
null rates:
 tenant_id           0.000000
tenant_name         0.000000
industry            0.000000
plan_tier           0.000000
region              0.000000
onboarded_at        0.000000
is_active           0.000000
security_contact    0.166667
cardinalities:
 tenant_id           6
tenant_name         6
industry            5
plan_tier           3
region              4
onboarded_at        5
is_active           2
security_contact    5
exact duplicate rows: 0
natural-key duplicate rows: 0 key: ['tenant_id']

agents.csv: rows=22, columns=12
null rates:
 agent_id                      0.0
release_version               0.0
agent_name                    0.0
agent_type                    0.0
model_id                      0.0
tenant_id                     0.0
sandbox_profile_id            0.0
owner_team                    0.0
autonomy_level                0.0
max_tool_calls_per_session    0.0
launch_date                   0.0
is_deprecated                 0.0
card

## Nested Sandbox Policy JSON: Before Cleaning

In [14]:
policy_objects = json.loads((ROOT / 'sandbox_policies.json').read_text(encoding='utf-8'))
policies_flat = pd.json_normalize(policy_objects, sep='.')
print('Policy versions:', len(policy_objects))
print('Flattened columns:', policies_flat.columns.tolist())
print('Null rates:\n', policies_flat.isna().mean().to_string())
print('Profile/version keys:\n', policies_flat[['sandbox_profile_id', 'policy_version', 'valid_from', 'valid_to']].to_string(index=False))
print('Policies missing network block:', sum('network' not in policy for policy in policy_objects))

Policy versions: 10
Flattened columns: ['sandbox_profile_id', 'policy_version', 'valid_from', 'valid_to', 'allowed_tools', 'escalation_allowed', 'network.egress_allowlist', 'network.denied_ports', 'network.max_egress_mb', 'filesystem.read_scope', 'filesystem.write_scope', 'resources.max_runtime_s', 'resources.max_memory_mb']
Null rates:
 sandbox_profile_id          0.0
policy_version              0.0
valid_from                  0.0
valid_to                    0.6
allowed_tools               0.0
escalation_allowed          0.0
network.egress_allowlist    0.1
network.denied_ports        0.1
network.max_egress_mb       0.1
filesystem.read_scope       0.0
filesystem.write_scope      0.0
resources.max_runtime_s     0.0
resources.max_memory_mb     0.0
Profile/version keys:
 sandbox_profile_id  policy_version           valid_from             valid_to
   sp_research_std               2 2026-01-01T00:00:00Z 2026-02-01T00:00:00Z
   sp_research_std               3 2026-01-25T00:00:00Z            

## Network Flows CSV: Before Cleaning

In [15]:
flows = profile_csv('network_flows.csv', ['flow_id'])
flow_time = pd.to_datetime(flows['flow_time'], format='mixed', utc=True, errors='coerce')
print('Flow timestamp parse failures:', flow_time.isna().sum())
print('Flow timestamp range:', flow_time.min(), 'to', flow_time.max())
print('Verdicts:\n', flows['verdict'].value_counts(dropna=False).to_string())
print('Protocols:\n', flows['protocol'].value_counts(dropna=False).to_string())
print('Destination ports:\n', flows['dst_port'].value_counts(dropna=False).to_string())


network_flows.csv: rows=42,392, columns=13
null rates:
 flow_id                0.000000
session_id             0.000000
sandbox_instance_id    0.000000
flow_time              0.000000
src_ip                 0.000000
dst_host               0.082893
dst_ip                 0.000000
dst_port               0.000000
protocol               0.000000
bytes_sent             0.000000
bytes_received         0.000000
verdict                0.000000
geo_country            0.049255
cardinalities:
 flow_id                42392
session_id             27858
sandbox_instance_id       28
flow_time              42308
src_ip                    28
dst_host                  12
dst_ip                    63
dst_port                   4
protocol                   4
bytes_sent              7362
bytes_received         35235
verdict                    3
geo_country                9
exact duplicate rows: 0
natural-key duplicate rows: 0 key: ['flow_id']
Flow timestamp parse failures: 0
Flow timestamp range: 2026-01-

## Nested SIEM Alert JSON: Before Cleaning

In [16]:
siem_doc = json.loads((ROOT / 'siem_alerts.json').read_text(encoding='utf-8'))
alerts = pd.json_normalize(siem_doc['alerts'], sep='.')
print('Envelope fields:', {key: value for key, value in siem_doc.items() if key != 'alerts'})
print('Alerts:', len(alerts))
print('Null rates:\n', alerts.isna().mean().to_string())
print('Alert ID duplicates:', alerts['alert_id'].duplicated().sum())
print('Source system counts:\n', alerts['source_system'].value_counts().to_string())
print('Severity counts:\n', alerts['severity'].value_counts().to_string())
alert_time = pd.to_datetime(alerts['detected_at'], format='mixed', utc=True, errors='coerce')
print('Detected-at parse failures:', alert_time.isna().sum(), 'range:', alert_time.min(), 'to', alert_time.max())
print('Observed timezone offsets:\n', alerts['detected_at'].astype(str).str.extract(r'([+-][0-9]{2}:[0-9]{2})$')[0].value_counts().to_string())
print('Alerts with source IP present among flow IPs:', alerts['source_ip'].isin(flows['src_ip']).sum(), '/', len(alerts))

Envelope fields: {'generated_at': '2026-04-01T02:00:00+02:00', 'source': 'corp-siem-export-v2'}
Alerts: 3754
Null rates:
 alert_id            0.0
source_system       0.0
severity            0.0
detected_at         0.0
source_ip           0.0
rule                0.0
confidence          0.0
destination.host    0.0
destination.port    0.0
technique.id        0.0
technique.name      0.0
Alert ID duplicates: 0
Source system counts:
 source_system
corp-siem       1912
cloud-ids       1251
egress-proxy     591
Severity counts:
 severity
high        1573
medium       962
low          728
critical     491
Detected-at parse failures: 0 range: 2026-01-31 01:19:25+00:00 to 2026-03-31 23:44:15+00:00
Observed timezone offsets:
 0
+02:00    1912
-05:00    1251
+03:00     591
Alerts with source IP present among flow IPs: 2503 / 3754


## Sandbox Audit Syslog: Before Cleaning

In [17]:
audit_pattern = re.compile(r'^(?P<mon>[A-Z][a-z]{2})\s+(?P<day>\d{1,2})\s+(?P<hms>\d{2}:\d{2}:\d{2})\s+(?P<host>\S+)\s+(?P<process>[^[]+)\[(?P<pid>\d+)\]:\s+session=(?P<session>\S+)\s+instance=(?P<instance>\S+)\s+decision=(?P<decision>\S+)\s+action=(?P<action>\S+)\s+target=(?P<target>\S+)\s+reason=(?P<reason>\S+)\s+policy_version=(?P<policy>\S+)\s*$')
audit_lines = (ROOT / 'sandbox_audit.log').read_text(encoding='utf-8', errors='replace').splitlines()
audit_matches = [audit_pattern.match(line) for line in audit_lines]
parsed_audit = [match.groupdict() for match in audit_matches if match]
print('Physical lines:', len(audit_lines), 'regex failures:', sum(match is None for match in audit_matches))
print('Parsed hosts:', len({row['host'] for row in parsed_audit}))
print('Decision counts:\n', pd.Series([row['decision'] for row in parsed_audit]).value_counts(dropna=False).to_string())
print('Action counts:\n', pd.Series([row['action'] for row in parsed_audit]).value_counts(dropna=False).to_string())

Physical lines: 184005 regex failures: 524
Parsed hosts: 42
Decision counts:
 ALLOW    177018
DENY       6463
Action counts:
 process_spawn     91171
file_write        48195
socket_connect    42179
file_read          1059
setuid              725
ptrace              152


## Findings Before Cleaning

The cells above are the evidence for row counts, schemas, nulls, cardinalities, duplicate keys, timestamp forms, and distributions. The observed defects include duplicate activity event IDs, unrecognized event types, unknown activity tenant/agent identifiers, multiple event-time encodings including malformed values, negative numeric values, invalid retry values, alias tool names, a duplicate tool catalogue row, unresolved tenant display-name similarity, incomplete pricing for the nano model, nullable flow destination metadata, malformed audit decisions, and SIEM alerts that do not all correspond to platform flows.

Cleaning decisions follow the project description: deduplicate by declared source key and keep earliest activity ingest time; resolve release-version prefix and documented tool-name aliases only; reject invalid/unparseable source records with one closed-enum reason; preserve optional missing values as null; infer the sandbox host-local offset from cross-source time alignment; apply policy validity windows at event time; and leave multi-candidate or absent SIEM-flow matches unlinked. No M2 labels or hold-out stream rows are used.

In [18]:
canonical_path = M1 / 'data' / 'canonical_activity.parquet'
rejects_path = M1 / 'data' / 'fact_rejected_record.csv'
metrics = json.loads((M1 / 'data' / 'metrics.json').read_text(encoding='utf-8'))
correlation = json.loads((M1 / 'data' / 'correlation_summary.json').read_text(encoding='utf-8'))

print('Canonical rows:', pq.ParquetFile(canonical_path).metadata.num_rows)
print('Rejected rows:', len(pd.read_csv(rejects_path)))
print('Correlation counts:', correlation)
print('Metrics:', json.dumps(metrics, indent=2, default=str))

Canonical rows: 1124728
Rejected rows: 43175
Correlation counts: {'rule': 'source_ip = src_ip AND destination.host = dst_ip or dst_host AND destination.port = dst_port AND |detected_at - flow_time| <= 60s (both in UTC)', 'tolerance_seconds': 60, 'matched': 2341, 'ambiguous': 25, 'unmatched': 1388, 'alerts_evaluated': 3754, 'coverage': 0.6236014917421417, 'ambiguous_policy': 'multi-candidate alerts are not linked', 'unmatched_policy': 'no candidate: not forced'}
Metrics: {
  "session_volume_terminal": 95046,
  "success_terminal_sessions": 87915,
  "error_terminal_sessions": 7131,
  "success_rate": 0.9249731708856764,
  "error_rate": 0.0750268291143236,
  "sessions_with_both_end_and_error": 0,
  "duration_sessions": 95046,
  "duration_p50_seconds": 17.635,
  "duration_p95_seconds": 39.613249999999994,
  "duration_successful_only_sessions": 87915,
  "duration_successful_only_p50_seconds": 17.459,
  "duration_successful_only_p95_seconds": 38.8463,
  "negative_duration_sessions": 0,
  "trac